# K-fold 결과로 YOLOv8s · YOLO11s · YOLO26s 비교

`run_kfold.py`가 모델마다 만든 `kfold_summary.csv` 3개를 한 번에 비교해요. seed를 여러 개 돌린 결과도 그대로 읽어요.

세 모델은 **똑같은 fold 나누기**로 학습했어요 (fold 나누기는 데이터와 split-seed로만 정해짐).
그래서 "fold0에서 v8 vs 11 vs 26"처럼 **같은 fold끼리 짝지어 비교**할 수 있어요. 그냥 평균만 비교하는 것보다 훨씬 공정해요.

**seed가 여러 개일 때 계산 방법:** 같은 fold의 seed 점수를 먼저 평균 내서 "fold 점수" 하나로 만들고, 그다음 fold끼리 평균과 표준편차를 내요.
이렇게 해야 seed를 많이 돌린 fold(예: fold0만 seed 3개)가 평균을 혼자 끌고 가지 않아요.

**순서**
1. 불러오기 (학습 실패 run 걸러내기)
2. 요약표 (평균 ± fold 표준편차)
3. 모델별 val 점수 막대
4. fold별 val 점수 (같은 fold끼리 비교)
5. 짝지은 차이: 1등 모델이 정말 나은가
6. seed 편차: 같은 fold에서 seed만 바꾸면 얼마나 흔들리나
7. 과적합 확인 (train − val 차이)
8. 정밀도와 재현율
9. 결론

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

# 노트북을 레포 루트(YacssoK/)에 두면 아래 경로 그대로 쓰면 돼요
DATA_DIR = Path('.')

# kfold_summary.csv 경로 -> 그래프에 표시할 모델 이름
# csv를 노트북 옆에 복사해 뒀다면 'kfold_summary8.csv'처럼 파일 이름만 써도 돼요
FILES = {
    'outputs/kfold/1026_modelA_yolov8s_v0_img640/kfold_summary.csv': 'YOLOv8s',
    'outputs/kfold/1027_sun_yolo11s_v0_img640/kfold_summary.csv': 'YOLO11s',
    'outputs/kfold/1027_sun_yolo26s_v0_img640/kfold_summary.csv': 'YOLO26s',
}

# 학습 실패 판정: train mAP50-95가 이보다 낮으면 "과적합"이 아니라 "학습이 제대로 안 된 run"으로 보고 평균에서 빼요
# (train 점수조차 낮다는 건 외우지도 못했다는 뜻. 과적합은 train은 높고 val만 낮은 경우예요)
FAIL_TRAIN_MAP = 0.80
EXCLUDE_FAILED = True   # False로 바꾸면 실패 run도 평균에 포함
COLORS = {'YOLOv8s': '#2a78d6', 'YOLO11s': '#eb6834', 'YOLO26s': '#1baf7a'}

GAP_WARN, GAP_BAD = 0.05, 0.10   # train-val 차이 판정 기준 (kfold_overfit.ipynb와 같음)

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e1e0d9',
    'grid.linewidth': 0.6,
    'axes.axisbelow': True,
    'axes.unicode_minus': False,
})
# 그래프 한글 폰트 (Mac: AppleGothic, Windows: Malgun Gothic)
import platform
from matplotlib import font_manager
KFONT = {'Darwin': 'AppleGothic', 'Windows': 'Malgun Gothic'}.get(platform.system(), 'NanumGothic')
if KFONT in {f.name for f in font_manager.fontManager.ttflist}:
    plt.rcParams['font.family'] = KFONT

## 1. 불러오기

In [ ]:
frames = []
for fname, model in FILES.items():
    df = pd.read_csv(DATA_DIR / fname)
    df['model'] = model
    frames.append(df)
df_all = pd.concat(frames, ignore_index=True)
MODELS = list(FILES.values())

df_all['failed'] = df_all['train_mAP50_95'] < FAIL_TRAIN_MAP
failed = df_all[df_all['failed']]
if len(failed):
    print(f'[학습 실패 의심] train mAP50-95 < {FAIL_TRAIN_MAP} 인 run {len(failed)}개'
          + (' → 평균에서 뺐어요' if EXCLUDE_FAILED else ' → 평균에 포함했어요'))
    display(failed[['model', 'fold', 'seed', 'train_mAP50_95', 'val_mAP50_95']].round(4))
    print('  해당 run의 yolo_run/results.csv에서 loss가 nan이 됐는지, 조기 종료됐는지 확인해 보세요.\n')
df = df_all[~df_all['failed']].copy() if EXCLUDE_FAILED else df_all.copy()

# seed 평균을 낸 fold 점수 (모델 1개 × fold 1개 = 1행)
fold_df = df.groupby(['model', 'fold'], as_index=False).mean(numeric_only=True)

# 같은 fold 나누기인지 확인: fold별 train/val 장수가 모델마다 같아야 짝지어 비교할 수 있어요
sizes = df.pivot_table(index='fold', columns='model', values='n_val')
same_split = (sizes.nunique(axis=1) == 1).all()
print('fold별 val 장수가 세 모델 모두 같음:', same_split)
print(f"모델 {df.model.nunique()}개, fold {df.fold.nunique()}개, seed {sorted(df_all.seed.unique().tolist())}")
display(df_all.pivot_table(index='fold', columns='model', values='seed', aggfunc='count')[MODELS]
        .rename_axis(columns='fold별 seed 수'))

## 2. 요약표

`평균 ± 표준편차`는 seed 평균을 낸 fold 점수 5개로 계산해요. 표준편차는 **fold 간 편차**예요. 어떤 조합이 val에 들어갔냐에 따라 점수가 이만큼 흔들린다는 뜻이에요.
`fold 1등 횟수`는 5개 fold 중 그 모델이 val mAP50-95 1등을 한 횟수예요.

In [ ]:
METRICS = {
    'val_mAP50_95': 'val mAP50-95',
    'val_mAP50': 'val mAP50',
    'val_precision': 'val precision',
    'val_recall': 'val recall',
    'train_mAP50_95': 'train mAP50-95',
    'gap_mAP50_95': 'train - val 차이',
}
agg = fold_df.groupby('model')[list(METRICS)].agg(['mean', 'std']).reindex(MODELS)
table = pd.DataFrame({
    label: agg[col]['mean'].map('{:.4f}'.format) + ' ± ' + agg[col]['std'].map('{:.4f}'.format)
    for col, label in METRICS.items()
})
wins = (fold_df.loc[fold_df.groupby('fold')['val_mAP50_95'].idxmax(), 'model']
        .value_counts().reindex(MODELS, fill_value=0))
table['fold 1등 횟수'] = wins.astype(str) + ' / ' + str(fold_df.fold.nunique())
display(table)

## 3. 모델별 val 점수

막대는 5개 fold 평균, 검은 선은 ± 표준편차, 점 하나하나는 각 fold 점수(seed 평균)예요.
막대 높이 차이보다 **점들이 얼마나 겹치는지**를 보세요. 많이 겹치면 차이가 확실하지 않다는 뜻이에요.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, title in [(axes[0], 'val_mAP50_95', 'val mAP50-95 (핵심 지표)'), (axes[1], 'val_mAP50', 'val mAP50')]:
    m, s = agg[col]['mean'], agg[col]['std']
    x = np.arange(len(MODELS))
    ax.bar(x, m, width=0.55, color=[COLORS[k] for k in MODELS], alpha=0.85)
    ax.errorbar(x, m, yerr=s, fmt='none', ecolor='#333', capsize=5, lw=1.2)
    for i, k in enumerate(MODELS):
        vals = fold_df.loc[fold_df.model == k, col].values
        jitter = np.linspace(-0.12, 0.12, len(vals))
        ax.scatter(i + jitter, vals, s=28, color='white', edgecolor='#333', lw=1, zorder=3)
    ax.set_xticks(x, [f'{k}\n평균 {m[k]:.3f}' for k in MODELS])  # 평균값은 축 이름 아래에 (점과 겹치지 않게)
    lo = fold_df[col].min()
    ax.set_ylim(max(0, lo - 0.03), 1.0)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 4. fold별 val 점수 (같은 fold끼리 비교)

선 하나가 모델 하나예요. 한 모델 선이 **모든 fold에서 위에** 있으면 확실히 나은 모델이고,
선끼리 서로 엇갈리면 "fold에 따라 이기는 모델이 바뀐다" → 실력 차이가 크지 않다는 뜻이에요.

In [ ]:
piv = fold_df.pivot_table(index='fold', columns='model', values='val_mAP50_95')[MODELS]
fig, ax = plt.subplots(figsize=(8, 4))
for k in MODELS:
    ax.plot(piv.index, piv[k], 'o-', color=COLORS[k], lw=2, ms=8, label=k)
ax.set_xticks(piv.index, [f'fold{f}' for f in piv.index])
ax.set_ylabel('val mAP50-95')
ax.set_title('fold별 val mAP50-95')
ax.legend()
plt.tight_layout()
plt.show()
display(piv.style.format('{:.4f}').highlight_max(axis=1, color='#cde2fb'))

## 5. 짝지은 차이: 1등 모델이 정말 나은가

평균이 가장 높은 모델을 기준으로, **같은 fold끼리** 다른 모델과의 점수 차이를 구해요.

- 차이가 5개 fold 모두 0보다 크면 → 기준 모델이 꾸준히 나음
- 0 위아래로 섞여 있으면 → 차이가 fold 운에 달려 있음
- `차이 평균 / 차이 표준편차`가 2보다 크면 꽤 믿을 만한 차이, 1보다 작으면 거의 구분이 안 되는 차이로 볼 수 있어요 (fold 5개라 대략적인 기준)

In [ ]:
best = agg['val_mAP50_95']['mean'].idxmax()
others = [k for k in MODELS if k != best]
diff = pd.DataFrame({k: piv[best] - piv[k] for k in others})

fig, ax = plt.subplots(figsize=(8, 3.8))
w = 0.35
x = np.arange(len(diff))
for i, k in enumerate(others):
    ax.bar(x + (i - 0.5) * w, diff[k], w, color=COLORS[k], label=f'{best} − {k}')
ax.axhline(0, color='#333', lw=1)
ax.set_xticks(x, [f'fold{f}' for f in diff.index])
ax.set_ylabel('val mAP50-95 차이')
ax.set_title(f'{best}가 같은 fold에서 얼마나 더 높은가 (0보다 위 = {best} 승)')
ax.legend()
plt.tight_layout()
plt.show()

rows = []
for k in others:
    d = diff[k]
    ratio = d.mean() / d.std() if d.std() > 0 else np.inf
    rows.append({
        '비교': f'{best} vs {k}',
        '차이 평균': d.mean(),
        '차이 표준편차': d.std(),
        '평균/표준편차': ratio,
        f'{best} 승': f'{(d > 0).sum()} / {len(d)}',
        '판단': '꽤 믿을 만한 차이' if ratio > 2 else ('약한 차이' if ratio > 1 else '구분 어려움'),
    })
pair = pd.DataFrame(rows)
display(pair.style.format({'차이 평균': '{:+.4f}', '차이 표준편차': '{:.4f}', '평균/표준편차': '{:.2f}'}))

## 6. seed 편차: 같은 fold에서 seed만 바꾸면 얼마나 흔들리나

seed를 2개 이상 돌린 fold만 그려요. 점 하나가 seed 하나, 가로 선이 seed 평균이에요.
학습 실패로 평균에서 뺀 run은 점수가 너무 낮아서 그래프 **맨 아래에 X와 실제 점수**로만 표시해요.

- 같은 모델의 점들이 넓게 퍼져 있으면 → 운(랜덤성)에 따라 점수가 크게 바뀌는 모델
- **seed 편차가 모델끼리의 점수 차이보다 크면**, 그 모델 차이는 운으로도 생길 수 있는 크기라서 "차이 없음"으로 보는 게 안전해요

In [ ]:
multi = df_all.groupby(['model', 'fold'])['seed'].count().unstack('model').reindex(columns=MODELS)
seed_folds = multi.index[(multi >= 2).any(axis=1)].tolist()

if not seed_folds:
    print('seed를 2개 이상 돌린 fold가 없어요. --seeds 42 43 44 로 돌린 뒤 다시 실행하세요.')
else:
    fig, axes = plt.subplots(1, len(seed_folds), figsize=(5.5 * len(seed_folds), 4.2), squeeze=False)
    for ax, f in zip(axes[0], seed_folds):
        # 축 범위는 정상 run 기준. 실패 run은 아래쪽 끝(floor)에 붙여서 표시
        lo = df[df.fold == f].val_mAP50_95.min() - 0.03
        floor = lo + 0.005
        for i, k in enumerate(MODELS):
            d = df_all[(df_all.model == k) & (df_all.fold == f)].sort_values('seed')
            ok, bad = d[~d.failed], d[d.failed]
            jitter = np.linspace(-0.18, 0.18, len(d)) if len(d) > 1 else np.zeros(1)
            jit = dict(zip(d.seed, jitter))
            ax.scatter([i + jit[sd] for sd in ok.seed], ok.val_mAP50_95, s=70, color=COLORS[k], edgecolor='white', lw=1.5, zorder=3)
            ax.scatter([i + jit[sd] for sd in bad.seed], [floor] * len(bad), s=70, color=COLORS[k], marker='x', lw=2, zorder=3)
            for _, r in ok.iterrows():
                ax.annotate(str(int(r.seed)), (i + jit[r.seed], r.val_mAP50_95), xytext=(6, 0),
                            textcoords='offset points', fontsize=7, color='#555', va='center')
            for _, r in bad.iterrows():
                ax.annotate(f'seed{int(r.seed)} 실패 ({r.val_mAP50_95:.2f})', (i + jit[r.seed], floor), xytext=(6, 0),
                            textcoords='offset points', fontsize=7, color='#c0392b', va='center')
            used = df[(df.model == k) & (df.fold == f)]
            if len(used):
                ax.hlines(used.val_mAP50_95.mean(), i - 0.25, i + 0.25, color=COLORS[k], lw=2.5)
        ax.set_xticks(range(len(MODELS)), MODELS)
        ax.set_xlim(-0.6, len(MODELS) - 0.4)
        ax.set_ylabel('val mAP50-95')
        ax.set_title(f'fold{f}: seed별 val mAP50-95')
        ax.set_ylim(lo, 1.0)
    plt.tight_layout()
    plt.show()

    rows = []
    for f in seed_folds:
        for k in MODELS:
            d = df[(df.model == k) & (df.fold == f)].val_mAP50_95
            n_fail = int(df_all[(df_all.model == k) & (df_all.fold == f)].failed.sum())
            rows.append({'fold': f, 'model': k, 'seed 수(실패 제외)': len(d), '실패 run': n_fail,
                         'seed 평균': d.mean(), 'seed 표준편차': d.std(), '최고-최저': d.max() - d.min() if len(d) else np.nan})
    seed_tab = pd.DataFrame(rows)
    display(seed_tab.style.format({'seed 평균': '{:.4f}', 'seed 표준편차': '{:.4f}', '최고-최저': '{:.4f}'}, na_rep='-'))

    for f in seed_folds:
        t = seed_tab[seed_tab.fold == f].dropna(subset=['seed 평균'])
        spread = t['seed 표준편차'].mean()
        model_diff = t['seed 평균'].max() - t['seed 평균'].min()
        print(f'fold{f}: 모델 간 seed 평균 차이 {model_diff:.4f}  vs  seed 표준편차 평균 {spread:.4f}'
              + ('  → seed 편차가 더 커서 모델 차이를 믿기 어려워요' if spread >= model_diff
                 else '  → 모델 차이가 seed 편차보다 커요'))

## 7. 과적합 확인 (train − val 차이)

왼쪽: fold별 train − val 차이. 노란 점선(0.05) 위면 약한 과적합, 빨간 점선(0.10) 위면 과적합이에요.
차이가 **음수**면 val이 train보다 높은 건데, val 사진이 38~39장으로 적어서 생기는 흔들림이라 걱정할 필요 없어요.

오른쪽: 점 하나가 (모델, fold) 하나예요. 대각선 위에 붙어 있을수록 train과 val 점수가 같다는 뜻 → 과적합 없음.

In [ ]:
gap = fold_df.pivot_table(index='fold', columns='model', values='gap_mAP50_95')[MODELS]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1.5, 1]})

ax = axes[0]
w = 0.26
x = np.arange(len(gap))
for i, k in enumerate(MODELS):
    ax.bar(x + (i - 1) * w, gap[k], w, color=COLORS[k], label=k)
ax.axhline(0, color='#333', lw=1)
ax.axhline(GAP_WARN, ls='--', c='#c98500', lw=1)
ax.axhline(GAP_BAD, ls='--', c='#c0392b', lw=1)
ax.text(len(gap) - 0.5, GAP_WARN, ' 주의 0.05', va='bottom', ha='right', fontsize=8, color='#c98500')
ax.text(len(gap) - 0.5, GAP_BAD, ' 과적합 0.10', va='bottom', ha='right', fontsize=8, color='#c0392b')
ax.set_xticks(x, [f'fold{f}' for f in gap.index])
ax.set_ylabel('train − val mAP50-95')
ax.set_title('fold별 train − val 차이')
ax.set_ylim(min(-0.05, gap.min().min() - 0.01), max(GAP_BAD + 0.02, gap.max().max() + 0.01))
ax.legend(fontsize=8)

ax = axes[1]
for k in MODELS:
    d = df[df.model == k]   # 여기는 seed 평균 전, run 하나하나
    ax.scatter(d['val_mAP50_95'], d['train_mAP50_95'], s=60, color=COLORS[k], edgecolor='white', lw=1.5, label=k, zorder=3)
lo = min(df['val_mAP50_95'].min(), df['train_mAP50_95'].min()) - 0.01
ax.plot([lo, 1], [lo, 1], color='#888', lw=1, ls='--')
ax.fill_between([lo, 1], [lo + GAP_BAD, 1 + GAP_BAD], [lo, 1], color='#c0392b', alpha=0.05)
ax.set_xlim(lo, 0.99)
ax.set_ylim(lo, 0.99)
ax.set_xlabel('val mAP50-95')
ax.set_ylabel('train mAP50-95')
ax.set_title('train vs val (대각선 = 차이 없음)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 8. 정밀도와 재현율

- **precision이 낮다** → 알약이 아닌 곳이나 다른 약으로 박스를 많이 친다 (오탐)
- **recall이 낮다** → 있는 알약을 놓친다 (미탐)

작은 점 하나가 run 하나, 큰 점이 모델 평균(요약표와 같은 값)이에요. 오른쪽 위로 갈수록 좋아요.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
for k in MODELS:
    d = df[df.model == k]
    ax.scatter(d['val_recall'], d['val_precision'], s=35, color=COLORS[k], alpha=0.45, edgecolor='none')
    # 큰 점은 요약표와 같은 값 (seed 평균 → fold 평균)
    ax.scatter(agg['val_recall']['mean'][k], agg['val_precision']['mean'][k], s=160, color=COLORS[k], edgecolor='white', lw=2, label=k, zorder=3)
ax.set_xlabel('val recall (놓치지 않는 정도)')
ax.set_ylabel('val precision (헛박스 안 치는 정도)')
ax.set_title('정밀도 vs 재현율')
ax.legend()
plt.tight_layout()
plt.show()

## 9. 결론 (숫자로 자동 정리)

In [ ]:
m = agg['val_mAP50_95']['mean']
s = agg['val_mAP50_95']['std']
g = agg['gap_mAP50_95']['mean']
gmax = fold_df.groupby('model')['gap_mAP50_95'].max().reindex(MODELS)

print('[성능] val mAP50-95 (평균 ± fold 표준편차)')
for k in m.sort_values(ascending=False).index:
    print(f'  {k:8s} {m[k]:.4f} ± {s[k]:.4f}   fold 1등 {wins[k]}회')

print('\n[1등과의 차이]')
for _, r in pair.iterrows():
    print(f"  {r['비교']}: 평균 {r['차이 평균']:+.4f}, {best} 승 {r[f'{best} 승']} → {r['판단']}")

print('\n[과적합] train − val 차이')
for k in MODELS:
    level = '과적합' if gmax[k] > GAP_BAD else ('약한 과적합 있는 fold 있음' if gmax[k] > GAP_WARN else '과적합 없음')
    print(f'  {k:8s} 평균 {g[k]:+.4f}, 가장 큰 fold {gmax[k]:+.4f} → {level}')

n_seed = df_all.groupby(['model', 'fold'])['seed'].count()
if len(failed):
    print(f'\n[학습 실패] {len(failed)}개 run을 ' + ('평균에서 뺐어요: ' if EXCLUDE_FAILED else '평균에 포함했어요: ')
          + ', '.join(f"{r.model} fold{r.fold} seed{r.seed}" for r in failed.itertuples()))
if n_seed.max() < 2:
    print('\n[주의] seed 1개 결과예요. 차이가 작으면 --seeds 42 43 44로 돌려서 seed 편차까지 확인하세요.')
elif n_seed.min() < 2:
    print('\n[주의] 일부 fold만 seed를 여러 개 돌렸어요. 나머지 fold도 돌리면 결론이 더 단단해져요.')